# Day 2 | Notebook 4a: Async Production Pipelines
**Author: Sattaya Singkul**

In high-traffic Python APIs (FastAPI), blocking the event loop is fatal. **AsyncSearchIndex** allows us to index and search without stopping other requests.

### Core Objectives:
1. **The Async Handshake**: Connecting to Redis via `redis.asyncio`.
2. **Parallel Ingestion**: Using `asyncio.gather` for high-throughput loads.
3. **Reactive Querying**: Non-blocking `await index.query()`.


In [2]:
import asyncio
import time
from redisvl.index import AsyncSearchIndex
from redisvl.schema import IndexSchema

redis_url = "redis://redis-vector-db:6379"

# 1. Define High-Performance Schema
schema = IndexSchema.from_dict({
    "index": {"name": "async_perf", "prefix": "perf:", "storage_type": "json"},
    "fields": [
        {"name": "text", "type": "text"},
        {"name": "embedding", "type": "vector", "attrs": {"dims": 3, "algorithm": "flat", "distance_metric": "cosine"}}
    ]
})

async def run_throughput_pipeline():
    index = AsyncSearchIndex(schema, redis_url=redis_url)
    await index.create(overwrite=True)
    
    # 🚀 HIGH-THROUGHPUT STRESS TEST (1,000 Records)
    batches = []
    for b in range(20):
        data = [{"text": f"Production Log Entry {b}-{i}", "embedding": [b/100, i/100, 0.5]} for i in range(50)]
        batches.append(index.load(data))
    
    start = time.time()
    await asyncio.gather(*batches)
    dur = time.time() - start
    
    print(f"✅ Stress Test: Loaded 1,000 documents across 20 parallel batches in {dur:.4f}s")
    
    # Async Search
    from redisvl.query import VectorQuery
    # return_fields specified
    vq = VectorQuery([0.1, 0.1, 0.5], "embedding", return_fields=["text"], num_results=1)
    res = await index.query(vq)
    print(f"✅ Verified High-Throughput Result: {res[0]['text']}")

await run_throughput_pipeline()


✅ Stress Test: Loaded 1,000 documents across 20 parallel batches in 0.1433s
✅ Verified High-Throughput Result: Production Log Entry 10-10


# Day 2 | Notebook 4b: Hybrid RRF Search Masterclass
**Author: Sattaya Singkul**

Semantic search is powerful, but keywords matter. **Hybrid Search** combines Vector similarity with Full-Text matching.

### Core Objectives:
1. **Boolean Logic + Vectors**: Strictly filtering semantic results.
2. **Reciprocal Rank Fusion (RRF)**: Merging different search results into a unified rank.
3. **The 'Exact ID' Edge Case**: Solving the failure of LLMs to find technical IDs.


In [3]:
from redisvl.query import VectorQuery
from redisvl.query.filter import Text

# THE USE CASE: User wants a "Macbook" (Exact keyword) but with "creative specs" (Semantic)
keyword_filter = Text("text") == "Macbook"
query_vec = [0.1, 0.2, 0.3] 

hybrid_query = VectorQuery(
    vector=query_vec,
    vector_field_name="embedding",
    filter_expression=keyword_filter, 
    num_results=3
)

print(f"Hybrid FT.SEARCH Command Gen:\n{str(hybrid_query)}")


Hybrid FT.SEARCH Command Gen:
@text:("Macbook")=>[KNN 3 @embedding $vector AS vector_distance] RETURN 1 vector_distance SORTBY vector_distance ASC DIALECT 2 LIMIT 0 3


# Day 2 | Notebook 4c: Persistent Agent Memory
**Author: Sattaya Singkul**

Web apps are stateless. AI Agents need **Semantic Memory** to remember context across refreshes.

### Core Objectives:
1. **SemanticSessionManager**: Storing chat turns as vectors.
2. **Long-Term Retrieval**: Fetching only the relevant history.
3. **Memory TTL**: Automatically clearing old secrets.


In [4]:
from redisvl.extensions.session_manager import SemanticSessionManager

session = SemanticSessionManager(
    name="agent_brain_v1",
    redis_url="redis://redis-vector-db:6379",
    distance_threshold=0.3
)

# Simulate Multi-Persona Conversations
user_history = {
    "gamer_pro": [
        {"role": "user", "content": "Looking for the best GPU for 4K gaming."},
        {"role": "assistant", "content": "The Aether-X1 with RTX 4090 is lead choice."}
    ],
    "creative_studio": [
        {"role": "user", "content": "I need a silent machine for color grading."},
        {"role": "assistant", "content": "The liquid-cooled Aether Pro is virtually silent under load."}
    ]
}

for user_id, messages in user_history.items():
    session.add_messages(messages)
    print(f"✅ Loaded history for: {user_id}")

# Contextual Retrieval (Recall gamer context)
context = session.get_recent(top_k=1)
print(f"\nLast Agent Context Trace: {context[0]['content']}")


/tmp/ipykernel_7469/329720378.py:1: DeprecationWarning: Importing from redisvl.extensions.session_manager is deprecated. StandardSessionManager has been renamed to MessageHistory. SemanticSessionManager has been renamed to SemanticMessageHistory. Please import from redisvl.extensions.message_history instead.
  from redisvl.extensions.session_manager import SemanticSessionManager


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Loaded history for: gamer_pro
✅ Loaded history for: creative_studio

Last Agent Context Trace: The liquid-cooled Aether Pro is virtually silent under load.


# Day 2 | Notebook 4d: Advanced Caching & Lifecycle
**Author: Sattaya Singkul**

Production RAG costs $0.00 if you cache the answer.

### Core Objectives:
1. **The Semantic Cache**: Caching complex JSON objects.
2. **Namespace Management**: Segregating caches by user/app.
3. **Invalidation Strategies**: Flushing based on data updates.


In [6]:
from redisvl.extensions.llmcache import SemanticCache

cache = SemanticCache(
    name="prod_cache",
    redis_url="redis://redis-vector-db:6379",
    distance_threshold=0.1
)

# 1. Cache a complex answer
cache.store("Specs for Macbook?", "M2 Chip, 16GB RAM, 512GB SSD", 
            vector=[0.1]*768, # Mock
            metadata={"source": "official_docs", "ts": time.time()})

print("✅ Cache Entry Provisioned")


Loading weights:   0%|          | 0/134 [00:00<?, ?it/s]

✅ Cache Entry Provisioned


# Day 2 | Notebook 4e: Semantic Intent Routing
**Author: Sattaya Singkul**

Don't call an expensive LLM just to classify a user's intent. Use Redis as a high-speed **Semantic Router**.

### Core Objectives:
1. **The Routing Table**: Mapping natural language to specific Tool IDs.
2. **Sub-1ms Classification**: Pure vector search for intent detection.
3. **Dynamic Routing**: Sending the query to different RAG pipelines based on the hit.


In [9]:
from redisvl.index import SearchIndex
from redisvl.query import VectorQuery

# 1. Define Expanded Intents (The Router Table)
intents = [
    {"intent": "technical_support", "example": "My laptop won't turn on and the screen is flickering."},
    {"intent": "technical_support", "example": "The GPU drivers are crashing during gameplay."},
    {"intent": "billing_query", "example": "I want a refund for my subscription."},
    {"intent": "billing_query", "example": "Where can I see my recent invoices?"},
    {"intent": "sales_inquiry", "example": "Do you have the Aether X5 in stock?"},
    {"intent": "sales_inquiry", "example": "What is the price of the Nebula Tablet?"},
    {"intent": "shipping_status", "example": "When will my package arrive in SF?"},
    {"intent": "account_security", "example": "I think my password was compromised."},
]

# 2. Index the Table
schema = {
    "index": {
        "name": "router", 
        "prefix": "route:",
        "storage_type": "json"
    }, 
    "fields": [
        {"name": "intent", "type": "tag"}, 
        {
            "name": "embedding", 
            "type": "vector", 
            "attrs": {
                "dims": 384,
                "algorithm": "flat",          # <--- Add this!
                "distance_metric": "cosine"   # <--- Add this too (best practice)
            }
        }
    ]
}
index = SearchIndex.from_dict(schema, redis_url="redis://redis-vector-db:6379")
index.create(overwrite=True)

from sentence_transformers import SentenceTransformer
model = SentenceTransformer('all-MiniLM-L6-v2')

data = [{"intent": i["intent"], "embedding": model.encode(i["example"]).tolist()} for i in intents]
index.load(data)

def route_request(query):
    q_vec = model.encode(query).tolist()
    vq = VectorQuery(q_vec, "embedding", return_fields=["intent"], num_results=1)
    res = index.query(vq)
    return res[0]['intent']

# TEST THE ROUTER WITH COMPLEX PHRASES
print(f"User: 'Package delayed in California' -> Route: {route_request('Package delayed in California')}")
print(f"User: 'Need to reset login credentials' -> Route: {route_request('Need to reset login credentials')}")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


User: 'Package delayed in California' -> Route: shipping_status
User: 'Need to reset login credentials' -> Route: account_security


# Day 2 | Notebook 4f: Index Observability & CLI Mastery
**Author: Sattaya Singkul**

Production engineers use the terminal to manage their Vector Databases. 

### Core Objectives:
1. **The `rvl` CLI**: Managing schemas from the shell.
2. **Memory Analytics**: Finding out exactly how much RAM your HNSW index is using.
3. **Index Statistics**: Monitoring index progress and failure rates.


In [10]:
# 1. Inspect existing indices
!rvl index list -u redis://redis-vector-db:6379

# 2. Check Memory Consumption of the PDR Index
!rvl index info -i pdr_idx -u redis://redis-vector-db:6379


usage: rvl index <command> [<args>]

Commands:
	info        Obtain information about an index
	create      Create a new index
	delete      Delete an existing index
	destroy     Delete an existing index and all of its data
	listall     List all indexes

positional arguments:
  command               Subcommand to run

options:
  -h, --help            show this help message and exit
  -i INDEX, --index INDEX
                        Index name
  -s SCHEMA, --schema SCHEMA
                        Path to schema file
  -u URL, --url URL     Redis URL
  --host HOST           Redis host
  -p PORT, --port PORT  Redis port
  --user USER           Redis username
  --ssl                 Use SSL
  -a PASSWORD, --password PASSWORD
                        Redis password


Index Information:
╭───────────────┬───────────────┬───────────────┬───────────────┬───────────────╮
│ Index Name    │ Storage Type  │ Prefixes      │ Index Options │ Indexing      │
├───────────────┼───────────────┼───────────────┼